# ADMET · Toxicity assessment for rapamycin and its rapalogs

Seventh in the series (potency → **A** → **D** → **M** → **E** → **T**oxicity). Same molecules.
Claims are grounded in primary literature (PubMed), cited **ACS style** in Step 6.

## Purpose

**Toxicity** asks: *is the drug safe* — cardiac (hERG), genotoxic (Ames), hepatotoxic (DILI), acutely
toxic (LD50)? These are the classic structure-based safety endpoints. A crucial twist for the rapalogs
is that their **real clinical toxicities are on-target** (mTOR inhibition), which structural models do
not describe at all.

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import io, zipfile, requests
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, rdFingerprintGenerator
from chembl_webresource_client.new_client import new_client
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, r2_score

# Notebook is stripped before push; each step's result is saved as a PNG here.
RESULTS = Path("../result_rapamycin_ADMET_Toxicity"); RESULTS.mkdir(exist_ok=True)
def save_fig(fig, name): fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight")
def save_img(obj, name):
    p = RESULTS / name
    obj.save(p) if hasattr(obj, "save") else p.write_bytes(obj.data)
def save_df_png(df, name, title=None):
    fig, ax = plt.subplots(figsize=(min(2 + len(df.columns) * 1.7, 18), 0.7 + 0.45 * len(df))); ax.axis("off")
    if title: ax.set_title(title, fontsize=10, loc="left")
    t = ax.table(cellText=df.astype(str).values, colLabels=list(df.columns), loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.4); t.auto_set_column_width(range(len(df.columns)))
    fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight"); plt.close(fig)

mol_client = new_client.molecule
RAPALOGS = {"Sirolimus (rapamycin)": "CHEMBL413", "Everolimus": "CHEMBL1908360",
            "Temsirolimus": "CHEMBL1201182", "Ridaforolimus": "CHEMBL2103839",
            "Zotarolimus": "CHEMBL219410"}
smiles = {n: mol_client.filter(molecule_chembl_id=c).only(["molecule_structures"])[0]
          ["molecule_structures"]["canonical_smiles"] for n, c in RAPALOGS.items()}
mols = {n: Chem.MolFromSmiles(s) for n, s in smiles.items()}

# Therapeutics Data Commons 'admet_group' benchmark (Huang et al. 2022), cached (git-ignored)
ZIP = Path("../data/tdc_admet_group.zip")
if not ZIP.exists():
    ZIP.write_bytes(requests.get("https://dataverse.harvard.edu/api/access/datafile/4426004",
                                 headers={"User-Agent": "Mozilla/5.0"}, timeout=120).content)
_zf = zipfile.ZipFile(ZIP)
_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def _bv(s):
    m = Chem.MolFromSmiles(s); return _GEN.GetFingerprint(m) if m else None
def _arr(bv):
    a = np.zeros((2048,), np.uint8); DataStructs.ConvertToNumpyArray(bv, a); return a
def _load(name):
    d = pd.concat([pd.read_csv(_zf.open(f"admet_group/{name}/{f}")) for f in ("train_val.csv", "test.csv")], ignore_index=True)
    return d[["Drug", "Y"]].dropna()
rap_bv = {n: _GEN.GetFingerprint(m) for n, m in mols.items()}

def run_models(models, pred_png, ad_png):
    "models: list of (label, tdc_dataset, task in clf/reg). Trains RF, predicts rapalogs + applicability domain."
    rows = {n.split(" ")[0]: {"rapalog": n.split(" ")[0]} for n in rap_bv}; cols = []
    for label, ds, task in models:
        d = _load(ds); bvs = [_bv(s) for s in d["Drug"]]
        keep = [i for i, b in enumerate(bvs) if b is not None]; bvs = [bvs[i] for i in keep]
        X = np.array([_arr(b) for b in bvs]); y = d["Y"].to_numpy()[keep]
        if task == "clf":
            m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
            proba = cross_val_predict(m, X, y, cv=5, method="predict_proba", n_jobs=-1)[:, 1]
            metric = f"ROC-AUC={roc_auc_score(y, proba):.2f}"; m.fit(X, y); pr = lambda a: round(m.predict_proba(a)[0, 1], 2)
        else:
            m = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            p = cross_val_predict(m, X, y, cv=5, n_jobs=-1); metric = f"R2={r2_score(y, p):.2f}"; m.fit(X, y); pr = lambda a: round(float(m.predict(a)[0]), 2)
        print(f"{label:26s} ({ds:28s}) n={len(y):5d}  CV {metric}")
        cols.append(label)
        for n, bv in rap_bv.items():
            k = n.split(" ")[0]
            rows[k][label] = pr(_arr(bv).reshape(1, -1))
            rows[k][label + " AD"] = round(max(DataStructs.BulkTanimotoSimilarity(bv, bvs)), 2)
    df = pd.DataFrame(rows.values())
    save_df_png(df, pred_png, "de-novo ML predictions + applicability domain (max Tanimoto to training set)")
    fig, ax = plt.subplots(figsize=(9.5, 4)); x = np.arange(len(df)); w = 0.8 / len(cols)
    for i, label in enumerate(cols):
        ax.bar(x + (i - (len(cols) - 1) / 2) * w, df[label + " AD"], w, label=label.split(" ")[0][:16])
    ax.axhline(0.3, color="red", ls="--", label="in-domain ~0.3")
    ax.set_xticks(x); ax.set_xticklabels(df["rapalog"], rotation=20)
    ax.set(ylabel="max Tanimoto to training set", ylim=(0, 1.05),
           title="Applicability domain of de-novo ML models for the rapalogs")
    ax.legend(fontsize=7, ncol=2); plt.tight_layout(); save_fig(fig, ad_png); plt.show()
    return df

print("Setup ready.")

## Step 1 — Honest caveat first (two caveats this time)

1. **The 900 Da problem, again.** Structural tox QSARs (hERG, Ames, DILI, LD50) are trained on small
   molecules; the rapalogs are **de-novo extrapolation** — the applicability domain is the verdict.
2. **The bigger caveat — the wrong kind of toxicity.** The rapalogs' actual, clinically important
   toxicities are **mechanism-based / on-target consequences of mTOR inhibition**: stomatitis/mucositis,
   non-infectious pneumonitis, metabolic effects (hyperlipidaemia, hyperglycaemia), myelosuppression,
   and immunosuppression → infections (Aapro et al., 2014). **None** of these are the structural-alert
   liabilities (hERG block, mutagenicity, reactive-metabolite hepatotoxicity) that the models below
   predict. So even a *perfect* structural tox model would miss what actually harms patients.

In [ ]:
tbl = pd.DataFrame({"MW (Da)": {n: round(Descriptors.MolWt(m)) for n, m in mols.items()},
                    "cLogP": {n: round(Descriptors.MolLogP(m), 1) for n, m in mols.items()}})
print(tbl.to_string())
save_df_png(tbl.reset_index().rename(columns={"index": "drug"}), "step1_properties.png", "Step 1 - the molecules")
grid = Draw.MolsToGridImage(list(mols.values()), legends=list(mols.keys()), molsPerRow=3, subImgSize=(300, 230))
save_img(grid, "step1_structures.png"); grid

## Step 2 — A rigorous in-silico toxicity stack (methods; not run here)

| Endpoint | In-silico model | Trustworthy for rapalogs? | Reference / data |
|---|---|---|---|
| **hERG** (cardiac) | ML classifier | ⚠️ out of domain (large molecule) | TDC (Huang et al., 2022) |
| **Ames** (mutagenicity) | ML classifier on ~6.5k benchmark | ⚠️ out of domain; rapalogs not alerting | Hansen et al., 2009 |
| **DILI** (hepatotoxicity) | ML classifier from structure | ⚠️ out of domain | Hammann et al., 2018 |
| **Acute toxicity (LD50)** | QSAR / consensus (CATMoS) | ⚠️ out of domain | Mansouri et al., 2021 (CATMoS); Zhu data |
| **On-target mTOR toxicity** | — (mechanism, not structure) | ❌ **not a structural QSAR endpoint** | Aapro et al., 2014 |

> **The rapalogs' dominant toxicity is on-target immunosuppressive/metabolic** — invisible to any
> structure-alert model. Structural QSARs answer "does this scaffold carry a classic liability?", not
> "what does inhibiting mTOR do to a patient?"

**What we run in Step 4:** hERG, Ames, DILI, and LD50 models (TDC data), predicted for the rapalogs
with applicability domain — then read against the on-target reality.

In [ ]:
measured = pd.DataFrame([
    {"toxicity": "Stomatitis / mucositis", "type": "on-target (mTOR)", "structural_alert?": "no", "ref": "Aapro 2014"},
    {"toxicity": "Non-infectious pneumonitis", "type": "on-target (mTOR)", "structural_alert?": "no", "ref": "Aapro 2014"},
    {"toxicity": "Hyperlipidaemia / hyperglycaemia", "type": "on-target (metabolic)", "structural_alert?": "no", "ref": "Aapro 2014"},
    {"toxicity": "Myelosuppression", "type": "on-target", "structural_alert?": "no", "ref": "Aapro 2014"},
    {"toxicity": "Immunosuppression -> infections", "type": "on-target (intended MoA)", "structural_alert?": "no", "ref": "Aapro 2014"},
    {"toxicity": "hERG / mutagenicity / classic DILI", "type": "structural (screened below)", "structural_alert?": "not prominent", "ref": "label / class"},
])
save_df_png(measured, "step3_measured_toxicity.png", "Step 3 - measured (clinical) toxicity: mostly on-target")
measured

## Step 3 — Measured toxicity (above) and Step 4 — run the ML stack

**Measured summary:** the rapalogs' clinically significant toxicities are **on-target mTOR effects**
(stomatitis, pneumonitis, metabolic, myelosuppression, infection risk) — **not** the structural-alert
liabilities the models predict. We still run the structural tox models to show *both* the
applicability-domain problem *and* the endpoint-mismatch problem.

In [ ]:
ml = run_models([("hERG block P", "herg", "clf"),
                 ("Ames mutagen P", "ames", "clf"),
                 ("DILI P", "dili", "clf"),
                 ("LD50 (log mol/kg)", "ld50_zhu", "reg")],
                "step4_ml_toxicity_predictions.png", "step4_applicability_domain.png")
ml

### What the run shows, and caveats (Step 5)

- **hERG:** predicted low block probability, and the rapalogs are **out of applicability domain**
  (large molecules) — consistent with hERG not being a prominent rapalog liability, but low-confidence.
- **Ames / DILI:** predicted low/uncertain; rapalogs are not classic genotoxins, but they are out of
  domain, so these are directional at best.
- **LD50:** an out-of-domain point estimate; not a substitute for measured toxicology.

**Caveats.** (i) 900 Da → structure-only tox QSAR is extrapolation; the AD column is the verdict.
(ii) **The endpoint mismatch is the real lesson:** the rapalogs' actual toxicities are **on-target
mTOR consequences** (Aapro et al., 2014), which no structural-alert model can predict. A clean "no
structural liability" readout is *not* a clean safety profile. (iii) **Measured / mechanism-based
toxicology is decisive** — structural QSAR is a narrow screen for a narrow set of liabilities.

## Step 6 — Citations (ACS style)

Bibliographic metadata retrieved from **PubMed**.

1. Aapro, M.; Andre, F.; Blackwell, K.; Calvo, E.; Jahanzeb, M.; Papazisis, K.; Porta, C.; Pritchard,
   K.; Ravaud, A. Adverse Event Management in Patients with Advanced Cancer Receiving Oral Everolimus:
   Focus on Breast Cancer. *Ann. Oncol.* **2014**, *25* (4), 763–773.
   https://doi.org/10.1093/annonc/mdu021.
2. Hansen, K.; Mika, S.; Schroeter, T.; Sutter, A.; ter Laak, A.; Steger-Hartmann, T.; Heinrich, N.;
   Müller, K.-R. Benchmark Data Set for in Silico Prediction of Ames Mutagenicity. *J. Chem. Inf.
   Model.* **2009**, *49* (9), 2077–2081. https://doi.org/10.1021/ci900161g.
3. Hammann, F.; Schöning, V.; Drewe, J. Prediction of Clinically Relevant Drug-Induced Liver Injury
   from Structure Using Machine Learning. *J. Appl. Toxicol.* **2019**, *39* (3), 412–419.
   https://doi.org/10.1002/jat.3741.
4. Mansouri, K.; Karmaus, A. L.; Fitzpatrick, J.; et al. CATMoS: Collaborative Acute Toxicity Modeling
   Suite. *Environ. Health Perspect.* **2021**, *129* (4), 047013. https://doi.org/10.1289/EHP8495.
5. Huang, K.; Fu, T.; Gao, W.; Zhao, Y.; Roohani, Y.; Leskovec, J.; Coley, C. W.; Xiao, C.; Sun, J.;
   Zitnik, M. Artificial Intelligence Foundation for Therapeutic Science. *Nat. Chem. Biol.* **2022**,
   *18* (10), 1033–1036. https://doi.org/10.1038/s41589-022-01131-2.

*Attribution: PubMed. Training data are the TDC `admet_group` benchmark (herg, ames, dili, ld50_zhu).*